<a href="https://colab.research.google.com/github/kasrasa/Object-detection-tutorial/blob/YOLO/YOLO_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U pycocotools
!pip install -q -U ultralytics

In [ ]:
import os
import random
import shutil
import urllib.request
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_iou
from torchvision.transforms.functional import to_tensor
from ultralytics import YOLO

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from pycocotools.coco import COCO
from IPython.display import display

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# -------------------------
# Global configuration
# -------------------------

SEED = 42
DEVICE = 0 if torch.cuda.is_available() else "cpu"  # Ultralytics accepts GPU index or "cpu"
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# COCO annotation paths uploaded in Colab.
TRAIN_ANN = "/content/data/train/instances_train2014.json"
VAL_ANN = "/content/data/valid/instances_val2014.json"
TRAIN_INSTANCES_ROOT = Path("/content/data/train/")
VAL_INSTANCES_ROOT = Path("/content/data/valid/")
TRAIN_INSTANCES_ROOT.mkdir(parents=True, exist_ok=True)
VAL_INSTANCES_ROOT.mkdir(parents=True, exist_ok=True)

# Image roots. Images are downloaded on demand into these folders.
IMAGE_ROOT = Path("/content/data/images")
LABEL_ROOT = Path("/content/data/labels")
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
LABEL_ROOT.mkdir(parents=True, exist_ok=True)
IMAGE_ROOT_EXPANDED = Path("/content/data/images_expanded")
LABEL_ROOT_EXPANDED = Path("/content/data/labels_expanded")
IMAGE_ROOT_EXPANDED.mkdir(parents=True, exist_ok=True)
LABEL_ROOT_EXPANDED.mkdir(parents=True, exist_ok=True)

# YOLO-native exported dataset root.
YOLO_DATA_ROOT = Path("/content/data/")
YOLO_ORIGINAL_ROOT = YOLO_DATA_ROOT / "original"
YOLO_EXPANDED_ROOT = YOLO_DATA_ROOT / "expanded"

# Dataset sizes.
NUM_TRAIN = 500
NUM_VAL = 200
MIN_SMALL_OBJECTS_PER_IMAGE = 3
NUM_ADDED_HARD_IMAGES = 100

# YOLO model and training settings.
# Change to "yolo11n.pt" or "yolov8n.pt" if you want an older baseline.
YOLO_WEIGHTS = "yolo26n.pt"
IMG_SIZE = 640
BATCH_SIZE = 8
NUM_WORKERS = 4
NUM_EPOCHS = 10
PATIENCE = 0
FREEZE_LAYERS = None  # Example: 10 to freeze early layers. None means no freeze argument.

# Evaluation / mining settings.
IOU_THRESH = 0.5 # regular iou threshold to match detected bbs to ground truth
SCORE_THRESH = 0.05 # intentionally low to see if model can detect objects with low confidence or completely misses them
POOR_RECALL_THRESHOLD = 0.7 # used to find weak classes and candidate classes for hard mining
MIN_SMALL_GT = 3 # number of small gt objects in the image that is not ocluded or crowded

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Using Ultralytics device:", DEVICE)
print("Train annotations exist:", os.path.exists(TRAIN_ANN))
print("Val annotations exist:", os.path.exists(VAL_ANN))

In [ ]:
coco_train = COCO(TRAIN_ANN)
coco_val = COCO(VAL_ANN)

In [ ]:
def build_coco_yolo_category_maps(coco):
  coco_to_yolo = {}
  yolo_to_coco = {}
  class_names = []
  cat_ids = sorted(coco.getCatIds())
  categories = coco.loadCats(cat_ids)
  for idx, cat in enumerate(categories):
    coco_to_yolo[cat["id"]] = idx # coco ids mapped to yolo
    yolo_to_coco[idx] = cat["id"] # yolo ids mapped to coco
    class_names.append(cat["name"])
  return coco_to_yolo, yolo_to_coco, class_names

def validate_ann(ann):
  if ann["iscrowd"] == 1:
    return False

  x, y, w, h = ann["bbox"]
  if ann.get("area", w*h) <= 1:
    return False
  if ann["bbox"][2] <= 1 or ann["bbox"][3] <= 1:
    return False
  return True

def coco_ann_to_yolo_row(ann, img_w, img_h, coco_to_yolo):
    x, y, w, h = ann["bbox"]

    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    bw = w / img_w
    bh = h / img_h

    class_id = coco_to_yolo[ann["category_id"]]

    return [class_id, cx, cy, bw, bh]

def create_yolo_dataset(anns, coco, coco_to_yolo):
  yolo_dataset = defaultdict(list)

  for ann in anns:
    image_id = ann["image_id"]
    image_info = coco.loadImgs(image_id)[0]

    img_w, img_h = image_info["width"], image_info["height"]
    yolo_row = coco_ann_to_yolo_row(ann, img_w, img_h, coco_to_yolo)
    yolo_dataset[image_id].append(yolo_row)
  return yolo_dataset

coco_to_yolo, yolo_to_coco, class_names = build_coco_yolo_category_maps(coco_train)
anns = coco_train.loadAnns(coco_train.getAnnIds())
yolo_dataset_train = create_yolo_dataset(anns, coco_train, coco_to_yolo)
coco_to_yolo, yolo_to_coco, class_names = build_coco_yolo_category_maps(coco_val)
anns = coco_val.loadAnns(coco_val.getAnnIds())
yolo_dataset_val = create_yolo_dataset(anns, coco_val, coco_to_yolo)
print("train images:", len(yolo_dataset_train))
print("val images:", len(yolo_dataset_val))

In [ ]:
def classify_bb_area(anns):
  buckets = {
      "small": defaultdict(list),
      "medium": defaultdict(list),
      "large": defaultdict(list),
      "all": defaultdict(list)
  }

  for ann in anns:
    if not validate_ann(ann):
      continue

    image_id = ann["image_id"]
    area = ann["area"]

    if area < 32*32:
      buckets["small"][image_id].append(ann)
    elif area < 96*96:
      buckets["medium"][image_id].append(ann)
    else:
      buckets["large"][image_id].append(ann)

    buckets["all"][image_id].append(ann)
  return buckets

def get_images_by_object_size(
    coco,
    size_bucket,
    category_ids=None,
    min_objects=MIN_SMALL_OBJECTS_PER_IMAGE,
):
    selected_image_ids = []

    image_ids = coco.getImgIds()
    ann_ids = coco.getAnnIds(imgIds=image_ids, iscrowd=False)
    anns = coco.loadAnns(ann_ids)

    buckets = classify_bb_area(anns)

    for image_id, bucket_anns in buckets[size_bucket].items():
      if len(bucket_anns) >= min_objects:
        for ann in bucket_anns:
          if category_ids is not None and ann["category_id"] not in category_ids:
            continue

          selected_image_ids.append(image_id)

    return selected_image_ids

def train_val_images(selected_image_ids_train, selected_image_ids_val, num_train, num_val, seed = SEED):
  random.seed(seed)

  train_image_ids = []
  val_image_ids = []

  train_image_ids = random.sample(selected_image_ids_train, min(num_train, len(selected_image_ids_train)))
  val_image_ids = random.sample(selected_image_ids_val, min(num_val, len(selected_image_ids_val)))

  return train_image_ids, val_image_ids

selected_image_ids = get_images_by_object_size(coco_train, "all")
print("selected images:", len(selected_image_ids))
print(selected_image_ids[:10])
yolo_dataset_train[selected_image_ids[0]]

selected_image_ids_train = get_images_by_object_size(coco_train, "all", min_objects=MIN_SMALL_OBJECTS_PER_IMAGE)
selected_image_ids_val = get_images_by_object_size(coco_val, "all", min_objects=MIN_SMALL_OBJECTS_PER_IMAGE)
selected_image_ids_train, selected_image_ids_val = train_val_images(
    selected_image_ids_train, selected_image_ids_val,
    NUM_TRAIN, NUM_VAL
)
print("train images:", len(selected_image_ids_train))
print("val images:", len(selected_image_ids_val))

In [ ]:
def create_yolo_txt_files(label_path, coco, image_ids, yolo_dataset, split = "train"):
  dataset_root = Path(label_path)

  label_dir = dataset_root / split

  label_dir.mkdir(parents=True, exist_ok=True)

  for image_id in image_ids:
    image_info = coco.loadImgs(image_id)[0]
    file_name = image_info["file_name"]

    label_file = label_dir / file_name.replace(".jpg", ".txt")

    yolo_rows = yolo_dataset.get(image_id,[])
    if len(yolo_rows) == 0:
      print(f"Warning: no labels for image_id={image_id}, file={file_name}")

    with open(label_file, "w", encoding="utf-8") as f:
      for row in yolo_rows:
        class_id, cx, cy, bw, bh = row
        f.write(
                    f"{int(class_id)} "
                    f"{cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n"
                )

def download_image_file(image_path, coco, image_ids, split = "train"):
  dataset_root = Path(image_path)

  image_dir = dataset_root / split

  image_dir.mkdir(parents=True, exist_ok=True)

  for image_id in image_ids:
    image_info = coco.loadImgs(image_id)[0]
    file_name = image_info["file_name"]

    image_file = image_dir / file_name

    if not image_file.exists():
      image_url = f"http://images.cocodataset.org/{split}2014/{file_name}"
      urllib.request.urlretrieve(image_url, image_file)


def write_yolo_yaml(dataset_path, class_names,expanded=False):
  root = Path(dataset_path)
  if expanded:
    filename = "data_expanded.yaml"
  else:
    filename = "data.yaml"

  with open(root/filename, "w", encoding="utf-8") as f:
    f.write(
        f"path: {dataset_path}\n"
        f"train: images/train\n"
        f"val: images/val\n"
        f"nc: {len(class_names)}\n"
        f"names: {class_names}\n"
    )

create_yolo_txt_files(LABEL_ROOT, coco_train, selected_image_ids_train, yolo_dataset_train)
create_yolo_txt_files(LABEL_ROOT, coco_val, selected_image_ids_val, yolo_dataset_val, "val")
download_image_file(IMAGE_ROOT, coco_train, selected_image_ids_train)
download_image_file(IMAGE_ROOT, coco_val, selected_image_ids_val, "val")
_, _, class_names = build_coco_yolo_category_maps(coco_train)
write_yolo_yaml(YOLO_DATA_ROOT, class_names)
label_files = list((Path(LABEL_ROOT) / "train").glob("*.txt"))
print("label files:", len(label_files))

first_label = label_files[0]
print("first label file:", first_label)

with open(first_label, "r") as f:
    print(f.read())

In [ ]:
model = YOLO(YOLO_WEIGHTS)
outputs = model.train(data = "data/data.yaml",
                      epochs = NUM_EPOCHS,
                      imgsz = IMG_SIZE,
                      batch = BATCH_SIZE,
                      patience = PATIENCE,
                      workers = NUM_WORKERS,
                      pretrained = True,
                      freeze = FREEZE_LAYERS,
                      device = -1,
                      seed = SEED)

results = model.predict(
    source="/content/data/images/val",
    imgsz=IMG_SIZE,
    conf=SCORE_THRESH,
    iou=0.7,
    device=-1,
    verbose=False,
)

In [ ]:
def yolo_pred_results(result):
  image_path = Path(result.path)
  file_name = image_path.name

  image_id = int(image_path.stem.split("_")[-1])

  height, width = result.orig_shape

  if result.boxes is None or len(result.boxes) == 0:
        boxes = torch.empty((0, 4))
        labels = torch.empty((0,), dtype=torch.long)
        scores = torch.empty((0,))
  else:
      boxes = result.boxes.xyxy.cpu()
      labels = result.boxes.cls.cpu().long()
      scores = result.boxes.conf.cpu()

  return {
      "image_id": image_id,
      "file_name": file_name,
      "image_path": image_path,
      "width": width,
      "height": height,
      "boxes": boxes,
      "labels": labels,
      "scores": scores,
  }

def create_results_dict(results):
  results_dict = {}

  for r in results:
    image_row = yolo_pred_results(r)

    results_dict[image_row["image_id"]] = image_row

  return results_dict

In [ ]:
preds_by_image_id = create_results_dict(results)

print("num images:", len(preds_by_image_id))

first_image_id = list(preds_by_image_id.keys())[0]
first_pred = preds_by_image_id[first_image_id]

print("image_id:", first_image_id)
print("file_name:", first_pred["file_name"])
print("width:", first_pred["width"])
print("height:", first_pred["height"])
print("boxes shape:", first_pred["boxes"].shape)
print("labels shape:", first_pred["labels"].shape)
print("scores shape:", first_pred["scores"].shape)

print(first_pred["boxes"][:5])
print(first_pred["labels"][:5])
print(first_pred["scores"][:5])

In [ ]:
def create_single_coco_gt_dict(coco_to_yolo, coco, image_id):
  image_info = coco.loadImgs([image_id])[0]

  file_name = image_info["file_name"]
  width = image_info["width"]
  height = image_info["height"]

  ann_ids = coco.getAnnIds(imgIds=[image_id], iscrowd=False)
  anns = coco.loadAnns(ann_ids)

  boxes = []
  labels = []
  areas = []
  size_buckets = []

  for ann in anns:
    if not validate_ann(ann):
      continue

    if ann["category_id"] not in coco_to_yolo:
      continue

    area = ann["area"]
    x,y,w,h = ann["bbox"]
    x1, y1 = x, y
    x2, y2 = x+w, y+h
    box = x1, y1, x2, y2

    label = coco_to_yolo[ann["category_id"]]

    boxes.append(box)
    labels.append(label)
    areas.append(area)
    size_buckets.append(
        "small" if area < 32*32 else "medium" if area < 96*96 else "large"
    )

  if len(boxes) == 0:
      boxes = torch.empty((0, 4), dtype=torch.float32)
      labels = torch.empty((0,), dtype=torch.long)
      areas = torch.empty((0,), dtype=torch.float32)
  else:
      boxes = torch.tensor(boxes, dtype=torch.float32)
      labels = torch.tensor(labels, dtype=torch.long)
      areas = torch.tensor(areas, dtype=torch.float32)

  return {
      "image_id": image_id,
      "file_name": file_name,
      "width": width,
      "height": height,
      "boxes": boxes,
      "labels": labels,
      "areas": areas,
      "size_buckets": size_buckets,
  }


def create_coco_gt_dict(coco_to_yolo, coco, image_ids):
  coco_gt_dict = {}

  for image_id in image_ids:
    coco_gt_dict[image_id] = create_single_coco_gt_dict(coco_to_yolo, coco, image_id)

  return coco_gt_dict



coco_gt_dict = create_coco_gt_dict(
    coco_to_yolo=coco_to_yolo,
    coco=coco_val,
    image_ids=selected_image_ids_val,
)
first_id = selected_image_ids_val[0]
gt = coco_gt_dict[first_id]

print("image_id:", gt["image_id"])
print("file_name:", gt["file_name"])
print("width:", gt["width"])
print("height:", gt["height"])
print("boxes shape:", gt["boxes"].shape)
print("labels shape:", gt["labels"].shape)
print("areas shape:", gt["areas"].shape)
print("num size buckets:", len(gt["size_buckets"]))

print(gt["boxes"][:5])
print(gt["labels"][:5])
print(gt["areas"][:5])
print(gt["size_buckets"][:5])

In [ ]:
def match_single_image(pred, gt, iou_thresh=IOU_THRESH):
    pred_boxes = pred["boxes"]
    pred_labels = pred["labels"]
    pred_scores = pred["scores"]

    gt_boxes = gt["boxes"]
    gt_labels = gt["labels"]
    gt_areas = gt["areas"]
    gt_size_buckets = gt["size_buckets"]

    image_id = gt["image_id"]

    image_matches = []

    matched_gt = set()
    matched_pred = set()

    if len(pred_boxes) == 0 or len(gt_boxes) == 0:
        return {
            "image_id": image_id,
            "matches": image_matches,
            "matched_pred_indices": matched_pred,
            "matched_gt_indices": matched_gt,
            "num_preds": len(pred_boxes),
            "num_gt": len(gt_boxes),
            "num_matched": 0,
        }

    ious = box_iou(pred_boxes, gt_boxes)


    pred_ordered = torch.argsort(pred_scores, descending=True)

    for pred_idx in pred_ordered:
        pred_idx = int(pred_idx)


        gt_ious = ious[pred_idx]
        gt_ordered = torch.argsort(gt_ious, descending=True)

        for gt_idx in gt_ordered:
            gt_idx = int(gt_idx)

            iou = float(ious[pred_idx, gt_idx])

            if iou < iou_thresh:
                break

            if gt_idx in matched_gt:
                continue

            if pred_labels[pred_idx].item() != gt_labels[gt_idx].item():
                continue

            matched_pred.add(pred_idx)
            matched_gt.add(gt_idx)

            image_matches.append(
                {
                    "image_id": image_id,
                    "pred_idx": pred_idx,
                    "gt_idx": gt_idx,
                    "iou": iou,
                    "label": int(gt_labels[gt_idx]),
                    "score": float(pred_scores[pred_idx]),
                    "area": float(gt_areas[gt_idx]),
                    "size_bucket": gt_size_buckets[gt_idx],
                }
            )

            break

    return {
        "image_id": image_id,
        "matches": image_matches,
        "matched_pred_indices": matched_pred,
        "matched_gt_indices": matched_gt,
        "num_preds": len(pred_boxes),
        "num_gt": len(gt_boxes),
        "num_matched": len(image_matches),
    }


def match_all_images(pred_dict, gt_dict, iou_thresh=IOU_THRESH):
    all_matches = {}

    for image_id, gt in gt_dict.items():
        pred = pred_dict[image_id]

        image_result = match_single_image(
            pred=pred,
            gt=gt,
            iou_thresh=iou_thresh,
        )

        all_matches[image_id] = image_result

    return all_matches

In [ ]:
all_matches = match_all_images(
    pred_dict=preds_by_image_id,
    gt_dict=coco_gt_dict,
    iou_thresh=IOU_THRESH,
)

first_id = selected_image_ids_val[0]
image_match = all_matches[first_id]

print("image_id:", image_match["image_id"])
print("num preds:", image_match["num_preds"])
print("num gt:", image_match["num_gt"])
print("num matched:", image_match["num_matched"])
print("matched pred indices:", image_match["matched_pred_indices"])
print("matches:")
print(image_match["matches"][:5])

In [ ]:
def size_bucket_matches_single_image(image_match, gt):
  image_id = gt["image_id"]

  stats = {
      "image_id": image_id,

      "small_matches": 0,
      "medium_matches": 0,
      "large_matches": 0,

      "small_missed": 0,
      "medium_missed": 0,
      "large_missed": 0,
  }

  matched_gt_indices = image_match["matched_gt_indices"]

  for gt_idx, size_bucket in enumerate(gt["size_buckets"]):
      if gt_idx in matched_gt_indices:
          stats[f"{size_bucket}_matches"] += 1
      else:
          stats[f"{size_bucket}_missed"] += 1

  return stats

def size_bucket_matches(all_matches, gt_dict):
  size_bucket_matches_dict = {}

  for image_id, image_match in all_matches.items():
    gt = gt_dict[image_id]

    size_bucket_dict = size_bucket_matches_single_image(image_match, gt)
    size_bucket_matches_dict[image_id] = size_bucket_dict

  return size_bucket_matches_dict

def summarize(size_bucket_matches):
  total = defaultdict(int)

  for image_id, size_bucket_dict in size_bucket_matches.items():
    for bucket in ["small", "medium", "large"]:
      total[f"{bucket}_matches"] += size_bucket_dict[f"{bucket}_matches"]
      total[f"{bucket}_missed"] += size_bucket_dict[f"{bucket}_missed"]

  summary = {}

  for bucket in ["small", "medium", "large"]:
      matches = total[f"{bucket}_matches"]
      missed = total[f"{bucket}_missed"]
      total_gt = matches + missed

      summary[bucket] = {
          "matches": matches,
          "missed": missed,
          "total_gt": total_gt,
          "recall": matches / total_gt if total_gt > 0 else None,
      }

  return summary

In [ ]:
def print_size_recall_summary(summary):
    for bucket in ["small", "medium", "large"]:
        row = summary[bucket]

        print(f"{bucket.upper()}")
        print(f"  matched: {row['matches']}")
        print(f"  missed:  {row['missed']}")
        print(f"  total:   {row['total_gt']}")
        print(f"  recall:  {row['recall']:.3f}")
        print()


size_stats_by_image = size_bucket_matches(
    all_matches,
    coco_gt_dict,
)

first_id = selected_image_ids_val[0]
print(size_stats_by_image[first_id])

total_recall = summarize(size_stats_by_image)
print_size_recall_summary(total_recall)

In [ ]:
def hard_samples_single_image(image_match, gt):
  image_id = gt["image_id"]
  file_name = gt["file_name"]

  matched_inds = image_match["matched_gt_indices"]

  hard_examples = []

  for gt_idx, size_bucket in enumerate(gt["size_buckets"]):
    if gt_idx in matched_inds:
      continue

    hard_examples.append({
        "image_id": image_id,
        "file_name": file_name,
        "gt_idx": gt_idx,
        "label": int(gt["labels"][gt_idx]),
        "area": float(gt["areas"][gt_idx]),
        "size_bucket": size_bucket,
        "box": gt["boxes"][gt_idx].tolist(),
    })

  return hard_examples

def hard_samples_total(all_matches, gt_dict):
  hard_images = {}

  for image_id, single_match in all_matches.items():
    gt = gt_dict[image_id]

    hard_examples = hard_samples_single_image(single_match, gt)

    if len(hard_examples) == 0:
      continue
    hard_images[image_id] = hard_examples

  return hard_images


hard_images = hard_samples_total(
    all_matches=all_matches,
    gt_dict=coco_gt_dict,
)
print(hard_images[first_id][:5])

In [ ]:
def score_coco_image_for_small_objects(coco, image_id, coco_to_yolo):
    ann_ids = coco.getAnnIds(imgIds=[image_id], iscrowd=False)
    anns = coco.loadAnns(ann_ids)

    num_small = 0
    num_medium = 0
    num_large = 0
    num_valid = 0
    labels = []

    for ann in anns:
        if not validate_ann(ann):
            continue

        coco_category_id = ann["category_id"]

        if coco_category_id not in coco_to_yolo:
            continue

        area = ann["area"]

        if area < 32 * 32:
            num_small += 1
        elif area < 96 * 96:
            num_medium += 1
        else:
            num_large += 1

        labels.append(coco_to_yolo[coco_category_id])

    score = 3 * num_small + 2 * num_medium + num_large

    return {
        "image_id": image_id,
        "score": score,
    }

def score_unused_train_images(coco, image_ids, coco_to_yolo):
    scored_images = []

    for image_id in image_ids:
        row = score_coco_image_for_small_objects(
            coco=coco,
            image_id=image_id,
            coco_to_yolo=coco_to_yolo,
        )

        scored_images.append(row)

    return scored_images

def select_top_hard_train_images(scored_images, top_k=300):
    scored_images = sorted(
        scored_images,
        key=lambda x: x["score"],
        reverse=True,
    )

    return scored_images[:top_k]

scored_unused = score_unused_train_images(
    coco=coco_train,
    image_ids=selected_image_ids_train,
    coco_to_yolo=coco_to_yolo,
)
hard_train_rows = select_top_hard_train_images(
    scored_images=scored_unused,
    top_k=300,
)

hard_train_ids = [row["image_id"] for row in hard_train_rows]


for row in hard_train_rows[:10]:
    print(row)

In [ ]:
expanded_train_ids = selected_image_ids_train + hard_train_ids
# print(f"num expanded train images: {len(expanded_train_ids)}")
# print(f"original train images: {len(selected_image_ids_train)}")

ann_ids = coco_train.getAnnIds(imgIds=expanded_train_ids, iscrowd=False)
anns = coco_train.loadAnns(ann_ids)
yolo_dataset_expanded = create_yolo_dataset(
    coco=coco_train,
    anns=anns,
    coco_to_yolo=coco_to_yolo,
)
create_yolo_txt_files(LABEL_ROOT, coco_train, expanded_train_ids, yolo_dataset_expanded)
download_image_file(IMAGE_ROOT, coco_train, expanded_train_ids)
_, _, class_names = build_coco_yolo_category_maps(coco_train)
write_yolo_yaml(YOLO_DATA_ROOT, class_names)
outputs = model.train(data = "data/data.yaml",
                      epochs = NUM_EPOCHS,
                      imgsz = IMG_SIZE,
                      batch = BATCH_SIZE,
                      patience = PATIENCE,
                      workers = NUM_WORKERS,
                      pretrained = True,
                      freeze = FREEZE_LAYERS,
                      device = -1,
                      seed = SEED)